# RTV Check-in Image Classifier — Full Pipeline

This notebook runs the complete ML pipeline from data download to model export:
1. **Setup** — Install dependencies
2. **Data Download** — Fetch dataset zip from Google Drive
3. **Task 1a** — Exploratory data analysis (all plots & statistics)
4. **Task 1b** — Data pipeline (split, augment, class weights, dataset.yaml)
5. **Task 2a** — Model training (YOLO11, single or two-phase)
6. **Task 2b** — Comprehensive evaluation (metrics, confusion matrix, error audit)
7. **Export** — Save trained model to Google Drive

> **Run in Google Colab** for free GPU. Set **Runtime → Change runtime type → T4 GPU**.

---
## 0. Setup & Install Dependencies

In [ ]:
!pip install -q ultralytics torch torchvision numpy pandas matplotlib seaborn scikit-learn Pillow opencv-python-headless tqdm gdown

In [ ]:
import os, sys, json, random, re, shutil, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm import tqdm

# Constants used throughout
SEED = 42
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
random.seed(SEED)
np.random.seed(SEED)

print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")

---
## 1. Download Dataset from Google Drive

**Two options:**
- **Option A:** Paste a Google Drive sharing URL below
- **Option B:** Mount Drive and set the path to the zip file

The sharing link should look like: `https://drive.google.com/file/d/XXXXXXX/view?usp=sharing`

In [ ]:
#@title >>> Paste your Google Drive sharing URL here
GDRIVE_URL = ""  #@param {type:"string"}

# === OR mount Google Drive directly ===
# Uncomment these 3 lines if the zip is already in your Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# GDRIVE_ZIP_PATH = "/content/drive/MyDrive/path/to/data.zip"  # <-- edit this path

In [ ]:
DATA_DIR = "data"
ZIP_PATH = "dataset.zip"

def extract_file_id(url):
    for pattern in [r'/file/d/([a-zA-Z0-9_-]+)', r'id=([a-zA-Z0-9_-]+)', r'/d/([a-zA-Z0-9_-]+)']:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Could not extract file ID from: {url}")

# Download
if GDRIVE_URL.strip():
    import gdown
    file_id = extract_file_id(GDRIVE_URL)
    print(f"File ID: {file_id}")
    gdown.download(f"https://drive.google.com/uc?id={file_id}&export=download",
                   ZIP_PATH, quiet=False, fuzzy=True)
    print(f"Downloaded: {os.path.getsize(ZIP_PATH)/1e6:.1f} MB")
elif 'GDRIVE_ZIP_PATH' in dir() and os.path.exists(GDRIVE_ZIP_PATH):
    shutil.copy2(GDRIVE_ZIP_PATH, ZIP_PATH)
    print(f"Copied from Drive: {os.path.getsize(ZIP_PATH)/1e6:.1f} MB")
else:
    print("Set GDRIVE_URL or mount Drive and set GDRIVE_ZIP_PATH above.")

In [ ]:
# Extract zip
if os.path.exists(ZIP_PATH):
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)

    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('.')

    # Handle zip structures — find the folder with category subdirs
    if not os.path.exists(DATA_DIR):
        expected = {'compost', 'pigsty', 'tippytap', 'vsla', 'organic'}
        for item in os.listdir('.'):
            if os.path.isdir(item) and item not in (DATA_DIR, '__MACOSX', '.config'):
                if expected & set(os.listdir(item)):
                    os.rename(item, DATA_DIR)
                    print(f"Renamed '{item}' -> '{DATA_DIR}'")
                    break

    # Cleanup macOS artifacts
    if os.path.exists('__MACOSX'):
        shutil.rmtree('__MACOSX')

    print("\nExtracted categories:")
    for d in sorted(os.listdir(DATA_DIR)):
        full = os.path.join(DATA_DIR, d)
        if os.path.isdir(full) and not d.startswith('.'):
            print(f"  {d:25s}: {len(os.listdir(full))} files")
else:
    print("No zip found — run the download cell first.")

---
## 2. Task 1a — Exploratory Data Analysis

Scan every image, collect metadata, and generate all analysis plots.

In [ ]:
# Scan dataset with corrupt image detection
records = []
corrupt_count = 0

for cat_dir in sorted(Path(DATA_DIR).iterdir()):
    if not cat_dir.is_dir() or cat_dir.name.startswith(('.', '_')):
        continue
    for img_file in sorted(cat_dir.iterdir()):
        if img_file.suffix.lower() not in IMG_EXTENSIONS:
            continue
        try:
            # Verify image integrity first
            with Image.open(img_file) as img:
                img.verify()
            # Re-open after verify (PIL requirement)
            with Image.open(img_file) as img:
                w, h = img.size
                records.append({
                    'category': cat_dir.name,
                    'filename': img_file.name,
                    'path': str(img_file),
                    'width': w,
                    'height': h,
                    'aspect_ratio': round(w / h, 3),
                    'channels': len(img.getbands()),
                    'color_mode': img.mode,
                    'file_size_kb': round(img_file.stat().st_size / 1024, 1),
                    'megapixels': round((w * h) / 1e6, 2),
                })
        except Exception as e:
            corrupt_count += 1
            print(f"[CORRUPT] {img_file}: {e}")

df = pd.DataFrame(records)
counts = df['category'].value_counts()
imbalance = counts.max() / counts.min()

print(f"\nTotal images:     {len(df)}")
print(f"Corrupt/skipped:  {corrupt_count}")
print(f"Categories:       {df['category'].nunique()}")
print(f"Imbalance ratio:  {imbalance:.1f}:1")
print(f"\nClass distribution:")
for cat in counts.index:
    print(f"  {cat:25s}: {counts[cat]:4d}  ({counts[cat]/len(df)*100:.1f}%)")

In [ ]:
# Plot 1: Class distribution
sorted_counts = counts.sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(sorted_counts.index, sorted_counts.values,
               color=sns.color_palette('viridis', len(sorted_counts)))
for bar, val in zip(bars, sorted_counts.values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontweight='bold')
ax.set_xlabel('Number of Images')
ax.set_title(f'Class Distribution (Imbalance Ratio: {imbalance:.1f}:1)')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Image dimensions scatter plot by category
fig, ax = plt.subplots(figsize=(10, 7))
categories = sorted(df['category'].unique())
palette = sns.color_palette('husl', len(categories))
for cat, color in zip(categories, palette):
    subset = df[df['category'] == cat]
    ax.scatter(subset['width'], subset['height'], label=cat,
               alpha=0.6, s=25, color=color)
ax.set_xlabel('Width (px)')
ax.set_ylabel('Height (px)')
ax.set_title('Image Dimensions by Category')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Aspect ratio histogram
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['aspect_ratio'], bins=40, color='steelblue', edgecolor='white')
ax.axvline(x=1.0, color='red', linestyle='--', label='Square (1:1)')
ax.set_xlabel('Aspect Ratio (W/H)')
ax.set_ylabel('Count')
ax.set_title('Aspect Ratio Distribution — informs padding vs crop strategy')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: File size box plot per category
order = df.groupby('category')['file_size_kb'].median().sort_values().index
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='file_size_kb', y='category', order=order,
            ax=ax, palette='viridis')
ax.set_xlabel('File Size (KB)')
ax.set_title('File Size Distribution by Category')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 5: Sample images grid
fig, axes = plt.subplots(len(categories), 4, figsize=(12, len(categories) * 2.2))
for i, cat in enumerate(categories):
    cat_df = df[df['category'] == cat]
    sampled = cat_df.sample(n=min(4, len(cat_df)), random_state=SEED)
    for j in range(4):
        ax = axes[i][j]
        ax.set_xticks([]); ax.set_yticks([])
        if j < len(sampled):
            img = Image.open(sampled.iloc[j]['path']).convert('RGB')
            ax.imshow(img)
        if j == 0:
            ax.set_ylabel(cat, fontsize=8, rotation=0, labelpad=75, ha='right')
fig.suptitle('Sample Images per Category', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("=" * 60)
print("  DATASET ANALYSIS SUMMARY")
print("=" * 60)
print(f"Total images:       {len(df)}")
print(f"Corrupt / skipped:  {corrupt_count}")
print(f"Classes:            {df['category'].nunique()}")
print(f"Imbalance ratio:    {imbalance:.1f}:1")
print(f"")
print(f"Width:   min={df['width'].min()}, max={df['width'].max()}, "
      f"mean={df['width'].mean():.0f}, std={df['width'].std():.0f}")
print(f"Height:  min={df['height'].min()}, max={df['height'].max()}, "
      f"mean={df['height'].mean():.0f}, std={df['height'].std():.0f}")
print(f"File KB: min={df['file_size_kb'].min()}, max={df['file_size_kb'].max()}, "
      f"mean={df['file_size_kb'].mean():.0f}")
print(f"Colour modes: {dict(df['color_mode'].value_counts())}")
print(f"")
print("Key observations:")
print(f"  1. Severe imbalance — guinea-pig-shelter has ~{counts.min()} images vs ~{counts.max()} for others")
print(f"  2. Visually similar classes: compost / organic / liquid-organic")
print(f"  3. Small dataset (~{len(df)} images) — transfer learning essential")
print(f"  4. Mixed resolutions — need standardised resize to 640x640")
print(f"  5. Field images: variable lighting, angles, backgrounds")

---
## 3. Task 1b — Data Pipeline (Split + Augmentation + YAML)

In [ ]:
OUTPUT_DIR = 'prepared_data'
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

# Collect paths and labels
paths  = df['path'].tolist()
labels = df['category'].tolist()

# Stratified two-step split
train_val_p, test_p, train_val_l, test_l = train_test_split(
    paths, labels, test_size=TEST_RATIO, stratify=labels, random_state=SEED)

val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_p, val_p, train_l, val_l = train_test_split(
    train_val_p, train_val_l, test_size=val_frac, stratify=train_val_l, random_state=SEED)

splits = {
    'train': list(zip(train_p, train_l)),
    'val':   list(zip(val_p, val_l)),
    'test':  list(zip(test_p, test_l)),
}

print("Split results:")
for name, data in splits.items():
    c = Counter(l for _, l in data)
    print(f"  {name:5s}: {len(data):4d} — {dict(sorted(c.items()))}")

In [ ]:
# Copy originals into YOLO directory structure
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

for split_name, data in splits.items():
    for img_path, label in data:
        dest_dir = os.path.join(OUTPUT_DIR, split_name, label)
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy2(img_path, os.path.join(dest_dir, os.path.basename(img_path)))

print("Copied originals into train/val/test.")

In [ ]:
# Augmentation functions (matching the full script set)
def _flip(img):    return img.transpose(Image.FLIP_LEFT_RIGHT)
def _rot_sm(img):  return img.rotate(random.uniform(-15, 15), fillcolor=(0,0,0))
def _rot_lg(img):  return img.rotate(random.uniform(-30, 30), fillcolor=(0,0,0))
def _bright(img):  return ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
def _contrast(img): return ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
def _color(img):   return ImageEnhance.Color(img).enhance(random.uniform(0.7, 1.3))
def _blur(img):    return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))
def _sharp(img):   return ImageEnhance.Sharpness(img).enhance(random.uniform(0.5, 2.0))

def _crop(img):
    w, h = img.size
    f = random.uniform(0.8, 1.0)
    nw, nh = int(w*f), int(h*f)
    left, top = random.randint(0, w-nw), random.randint(0, h-nh)
    return img.crop((left, top, left+nw, top+nh)).resize((w, h), Image.BILINEAR)

def _combined(img):
    if random.random() > 0.5: img = _flip(img)
    img = img.rotate(random.uniform(-10, 10), fillcolor=(0,0,0))
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))
    return img

AUG_FNS = [_flip, _rot_sm, _rot_lg, _bright, _contrast, _color, _blur, _sharp, _crop, _combined]

# Augment minority classes to ~80% of majority
train_counts = Counter(l for _, l in splits['train'])
target = int(max(train_counts.values()) * 0.8)

for label, count in train_counts.items():
    needed = target - count
    if needed <= 0:
        continue
    print(f"Augmenting {label}: +{needed} images (from {count} originals)")
    src_imgs = [p for p, l in splits['train'] if l == label]
    dest_dir = os.path.join(OUTPUT_DIR, 'train', label)
    for i in range(needed):
        src = random.choice(src_imgs)
        img = Image.open(src).convert('RGB')
        aug = AUG_FNS[i % len(AUG_FNS)](img)
        aug.save(os.path.join(dest_dir, f'aug_{i:04d}_{os.path.basename(src)}'), quality=95)

print("\nFinal training counts:")
for d in sorted(Path(OUTPUT_DIR, 'train').iterdir()):
    if d.is_dir():
        print(f"  {d.name:25s}: {len(list(d.glob('*')))}")

In [ ]:
# Compute and save class weights (inverse frequency)
train_labels = [l for _, l in splits['train']]
tc = Counter(train_labels)
total_train = sum(tc.values())
n_cls = len(tc)
class_weights = {lab: round(total_train / (n_cls * cnt), 4) for lab, cnt in tc.items()}

print("Class weights (inverse frequency):")
for lab in sorted(class_weights):
    print(f"  {lab:25s}: {class_weights[lab]:.3f}")

# Save as JSON
with open(os.path.join(OUTPUT_DIR, 'class_weights.json'), 'w') as f:
    json.dump(class_weights, f, indent=2)
print(f"\nSaved to {OUTPUT_DIR}/class_weights.json")

In [ ]:
# Create dataset.yaml (required by YOLO)
class_names = sorted(df['category'].unique())
yaml_lines = [
    '# RTV Check-in Image Classification Dataset',
    f'path: {os.path.abspath(OUTPUT_DIR)}',
    'train: train',
    'val: val',
    'test: test',
    '',
    f'nc: {len(class_names)}',
    '',
    'names:',
]
for i, name in enumerate(class_names):
    yaml_lines.append(f'  {i}: {name}')

yaml_path = os.path.join(OUTPUT_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    f.write('\n'.join(yaml_lines) + '\n')

print(f"Created {yaml_path}")
print(open(yaml_path).read())

---
## 4. Task 2a — Model Training (YOLO11)

In [ ]:
from ultralytics import YOLO

#@title >>> Training Configuration
MODEL_NAME = 'yolo11n-cls.pt'  #@param ['yolo11n-cls.pt', 'yolo11s-cls.pt', 'yolo11m-cls.pt']
EPOCHS     = 50   #@param {type:"integer"}
IMGSZ      = 640  #@param {type:"integer"}
BATCH      = 32   #@param {type:"integer"}
LR         = 0.001 #@param {type:"number"}
PATIENCE   = 15   #@param {type:"integer"}
TWO_PHASE  = True  #@param {type:"boolean"}

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print(f"Model:  {MODEL_NAME}")
print(f"Epochs: {EPOCHS} | Batch: {BATCH} | LR: {LR}")
print(f"Strategy: {'Two-phase (freeze then finetune)' if TWO_PHASE else 'Single-phase'}")

In [ ]:
if TWO_PHASE:
    # Phase 1: head only (backbone frozen)
    print("=" * 60)
    print("PHASE 1 — Classifier head only (backbone frozen, 10 epochs)")
    print("=" * 60)
    model = YOLO(MODEL_NAME)
    model.train(
        data=OUTPUT_DIR, epochs=10, imgsz=IMGSZ, batch=BATCH,
        lr0=0.01, lrf=0.1, patience=10, device=DEVICE,
        project='training', name='phase1_head', exist_ok=True,
        pretrained=True, optimizer='AdamW', freeze=10,
        warmup_epochs=2, cos_lr=True, seed=SEED, workers=2, verbose=True,
    )

    # Phase 2: full fine-tuning
    print("\n" + "=" * 60)
    print(f"PHASE 2 — Full fine-tuning ({EPOCHS - 10} epochs)")
    print("=" * 60)
    model = YOLO('training/phase1_head/weights/best.pt')
    model.train(
        data=OUTPUT_DIR, epochs=EPOCHS-10, imgsz=IMGSZ, batch=BATCH,
        lr0=LR, lrf=0.01, patience=PATIENCE, device=DEVICE,
        project='training', name='phase2_finetune', exist_ok=True,
        optimizer='AdamW', warmup_epochs=3, cos_lr=True, seed=SEED, workers=2,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=15.0, translate=0.1, scale=0.5,
        fliplr=0.5, mixup=0.1, erasing=0.3, verbose=True,
    )
    BEST_MODEL = 'training/phase2_finetune/weights/best.pt'

else:
    print("=" * 60)
    print(f"Single-phase training ({EPOCHS} epochs)")
    print("=" * 60)
    model = YOLO(MODEL_NAME)
    model.train(
        data=OUTPUT_DIR, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        lr0=LR, lrf=0.01, patience=PATIENCE, device=DEVICE,
        project='training', name='single_phase', exist_ok=True,
        pretrained=True, optimizer='AdamW', weight_decay=5e-4,
        warmup_epochs=5, cos_lr=True, seed=SEED, workers=2,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=15.0, translate=0.1, scale=0.5,
        fliplr=0.5, mixup=0.1, erasing=0.3, crop_fraction=0.8, verbose=True,
    )
    BEST_MODEL = 'training/single_phase/weights/best.pt'

print(f"\nBest model: {BEST_MODEL}")
print(f"Size: {os.path.getsize(BEST_MODEL)/1e6:.1f} MB")

---
## 5. Task 2b — Comprehensive Evaluation

In [ ]:
# YOLO built-in evaluation on test split
model = YOLO(BEST_MODEL)
val_results = model.val(split='test')

top1 = val_results.results_dict.get('metrics/accuracy_top1', 0)
top5 = val_results.results_dict.get('metrics/accuracy_top5', 0)
print(f"Top-1 Accuracy: {top1:.4f}")
print(f"Top-5 Accuracy: {top5:.4f}")

In [ ]:
# Per-image inference for detailed metrics
test_dir = Path(OUTPUT_DIR) / 'test'
class_names = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])

y_true, y_pred, confidences, predictions = [], [], [], []

for class_dir in sorted(test_dir.iterdir()):
    if not class_dir.is_dir(): continue
    true_label = class_dir.name
    for img_path in sorted(class_dir.iterdir()):
        if img_path.suffix.lower() not in IMG_EXTENSIONS: continue
        preds = model.predict(str(img_path), imgsz=IMGSZ, verbose=False)
        p = preds[0]
        pred_label = p.names[p.probs.top1]
        conf = float(p.probs.top1conf)
        top5_labels = [p.names[i] for i in p.probs.top5]

        y_true.append(true_label)
        y_pred.append(pred_label)
        confidences.append(conf)
        predictions.append({
            'path': str(img_path), 'true': true_label,
            'pred': pred_label, 'conf': conf,
            'correct': true_label == pred_label,
            'in_top5': true_label in top5_labels,
        })

print(f"Evaluated {len(predictions)} test images")

In [ ]:
# Classification report
print(classification_report(y_true, y_pred, labels=class_names, zero_division=0))

macro_f1 = f1_score(y_true, y_pred, labels=class_names, average='macro', zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, labels=class_names, average='weighted', zero_division=0)
top1_acc = sum(p['correct'] for p in predictions) / len(predictions)
top5_acc = sum(p['in_top5'] for p in predictions) / len(predictions)

print(f"Top-1 Accuracy: {top1_acc:.4f}")
print(f"Top-5 Accuracy: {top5_acc:.4f}")
print(f"Macro F1:       {macro_f1:.4f}")
print(f"Weighted F1:    {weighted_f1:.4f}")

In [ ]:
# Confusion matrix (raw + normalised)
cm = confusion_matrix(y_true, y_pred, labels=class_names)
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, data, fmt, title in [
    (axes[0], cm, 'd', 'Confusion Matrix (Counts)'),
    (axes[1], cm_norm, '.2f', 'Confusion Matrix (Normalised)'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 bar chart
report = classification_report(y_true, y_pred, labels=class_names,
                               output_dict=True, zero_division=0)
f1_scores = [report[c]['f1-score'] for c in class_names]
colours = ['#d32f2f' if s < 0.5 else '#ff9800' if s < 0.7 else '#388e3c' for s in f1_scores]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(class_names, f1_scores, color=colours, edgecolor='grey')
for bar, val in zip(bars, f1_scores):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.axvline(macro_f1, color='blue', ls='--', alpha=0.5, label=f'Macro F1 = {macro_f1:.3f}')
ax.set_xlim(0, 1.15)
ax.set_xlabel('F1 Score')
ax.set_title('Per-Class F1 Scores')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Confidence distribution
correct_c = [p['conf'] for p in predictions if p['correct']]
wrong_c   = [p['conf'] for p in predictions if not p['correct']]

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 1, 30)
ax.hist(correct_c, bins=bins, alpha=0.7, color='green',
        edgecolor='darkgreen', label=f'Correct (n={len(correct_c)})')
if wrong_c:
    ax.hist(wrong_c, bins=bins, alpha=0.7, color='red',
            edgecolor='darkred', label=f'Wrong (n={len(wrong_c)})')
ax.set_xlabel('Confidence')
ax.set_ylabel('Count')
ax.set_title('Prediction Confidence Distribution')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean confidence (correct): {np.mean(correct_c):.4f}")
print(f"Mean confidence (wrong):   {np.mean(wrong_c):.4f}" if wrong_c else "No errors!")

In [ ]:
# Misclassification audit
errors = [p for p in predictions if not p['correct']]
print(f"Total errors: {len(errors)} / {len(predictions)} ({len(errors)/len(predictions)*100:.1f}%)")

if errors:
    # Top confusion pairs
    pair_counts = Counter((p['true'], p['pred']) for p in errors)
    print("\nTop confusion pairs (true -> predicted):")
    for (true, pred), cnt in pair_counts.most_common(10):
        print(f"  {true:25s} -> {pred:25s} : {cnt}x")

    # Highest-confidence errors (most dangerous in production)
    print("\nHighest-confidence errors (most dangerous):")
    for p in sorted(errors, key=lambda x: -x['conf'])[:5]:
        print(f"  {p['true']:25s} predicted as {p['pred']:25s}  conf={p['conf']:.3f}")

    # Lowest-confidence correct (most fragile)
    correct_preds = [p for p in predictions if p['correct']]
    print("\nLowest-confidence correct (fragile):")
    for p in sorted(correct_preds, key=lambda x: x['conf'])[:5]:
        print(f"  {p['true']:25s}  conf={p['conf']:.3f}")
else:
    print("Perfect score — no misclassifications!")

In [ ]:
# Save metrics and predictions as JSON
metrics = {
    'top1_accuracy': top1_acc, 'top5_accuracy': top5_acc,
    'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
    'total_samples': len(predictions),
    'correct': sum(p['correct'] for p in predictions),
    'errors': len(errors),
    'per_class': report,
    'confusion_matrix': cm.tolist(),
    'class_names': class_names,
}

with open('evaluation_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)
with open('evaluation_predictions.json', 'w') as f:
    json.dump(predictions, f, indent=2)

print("Saved evaluation_metrics.json")
print("Saved evaluation_predictions.json")

---
## 6. Export Model to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title >>> Export destination
DRIVE_EXPORT_DIR = '/content/drive/MyDrive/rtv_classifier'  #@param {type:"string"}

os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)

# Copy best model
shutil.copy2(BEST_MODEL, os.path.join(DRIVE_EXPORT_DIR, 'best.pt'))
print(f"Model exported: {DRIVE_EXPORT_DIR}/best.pt ({os.path.getsize(BEST_MODEL)/1e6:.1f} MB)")

# Copy last model as backup
last = BEST_MODEL.replace('best.pt', 'last.pt')
if os.path.exists(last):
    shutil.copy2(last, os.path.join(DRIVE_EXPORT_DIR, 'last.pt'))
    print(f"Backup exported: {DRIVE_EXPORT_DIR}/last.pt")

# Copy evaluation files
for f in ['evaluation_metrics.json', 'evaluation_predictions.json']:
    if os.path.exists(f):
        shutil.copy2(f, os.path.join(DRIVE_EXPORT_DIR, f))

print(f"\nAll files in {DRIVE_EXPORT_DIR}:")
for f in sorted(os.listdir(DRIVE_EXPORT_DIR)):
    size = os.path.getsize(os.path.join(DRIVE_EXPORT_DIR, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

In [ ]:
# Alternative: download best.pt directly to your local machine
from google.colab import files
files.download(BEST_MODEL)

---
## 7. Next Steps — Local Deployment

After downloading `best.pt`:

```bash
# Place the model
mkdir -p outputs/training
cp ~/Downloads/best.pt outputs/training/best.pt

# Run FastAPI
export MODEL_PATH=outputs/training/best.pt
uvicorn src.app.main:app --host 0.0.0.0 --port 8000

# Or deploy with Docker
docker compose up --build

# Open http://localhost:8000 to test
```